In [12]:
!pip install gymnasium ale-py autorom opencv-python tensorflow

In [ ]:
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	C:\Users\ecarter\AppData\Local\anaconda3\Lib\site-packages\AutoROM\roms

Existing ROMs will be overwritten.


In [1]:
import ale_py
import gymnasium as gym
import time
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random

print("Imports done")

Imports done


In [2]:
""" LOAD SAVED MODEL """

# Load the model
online_net = tf.keras.models.load_model("kingkong_dqn.keras")
print("Model loaded from kingkong_dqn.keras")

Model loaded from kingkong_dqn.keras


In [3]:
""" TRAIN NEW MODEL """

gym.register_envs(ale_py)

env = gym.make("ALE/KingKong-v5", render_mode="rgb_array")
obs, info = env.reset()

# ── Hyperparameters ────────────────────────────────────────────────────────────
INPUT_SHAPE     = (84, 84, 1)
NUM_ACTIONS     = int(env.action_space.n)

TRAIN_SECONDS   = 300          # How long to train (seconds)
EVAL_SECONDS    = 30           # How long each evaluation window lasts
EVAL_EVERY      = 60           # Run an evaluation every N training seconds

GAMMA           = 0.99         # Discount factor
EPSILON_START   = 1.0          # Initial exploration rate
EPSILON_MIN     = 0.05         # Minimum exploration rate
EPSILON_DECAY   = 0.995        # Multiplicative decay applied each episode

BATCH_SIZE      = 32
REPLAY_CAPACITY = 10_000       # Experience replay buffer size
MIN_REPLAY_SIZE = 1_000        # Don't train until buffer has this many samples
LEARNING_RATE   = 1e-4
TARGET_UPDATE_EVERY = 500      # Steps between target-network syncs

# ── Custom Reward Shaping Hyperparameters ──────────────────────────────────────
ALPHA               = 0.05     # Weight on time cost in completion bonus
BETA                = 0.005    # Per-timestep penalty weight
SCREEN_BONUS        = 50.0     # Reward for advancing to a new screen
CHECKPOINT_BONUS    = 20.0     # Reward for hitting a mid-screen checkpoint
COMPLETION_BONUS    = 1000.0   # Base reward for completing the game
STALL_PENALTY       = 5.0      # Penalty for making no progress
STALL_THRESHOLD     = 200      # Steps without progress before stall penalty fires
REWARD_CLIP         = 1.0      # Clip shaped rewards to [-REWARD_CLIP, REWARD_CLIP]


# ── Networks ───────────────────────────────────────────────────────────────────
def build_q_network(input_shape, num_actions):
    """
    A small CNN Q-network.
    Outputs Q(s, a) for every action a simultaneously.
    """
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(16, 8, strides=4, activation="relu")(inputs)
    x = keras.layers.Conv2D(32, 4, strides=2, activation="relu")(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    outputs = keras.layers.Dense(num_actions, activation=None)(x)
    return keras.Model(inputs, outputs)

# Online network  — trained every step
online_net = build_q_network(INPUT_SHAPE, NUM_ACTIONS)
# Target network — updated periodically for stable TD targets
target_net = build_q_network(INPUT_SHAPE, NUM_ACTIONS)
target_net.set_weights(online_net.get_weights())

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
online_net.summary()
print("Networks created")


# ── Helpers ────────────────────────────────────────────────────────────────────
def preprocess_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return (small / 255.0).astype(np.float32)

def select_action(frame, epsilon):
    if np.random.rand() < epsilon:
        return env.action_space.sample()
    processed = preprocess_frame(frame)
    tensor    = processed[np.newaxis, ..., np.newaxis]          # (1,84,84,1)
    logits    = online_net(tensor, training=False)
    return int(tf.argmax(logits[0]).numpy())

replay_buffer = deque(maxlen=REPLAY_CAPACITY)

def store_transition(state, action, reward, next_state, done):
    replay_buffer.append((state, action, reward, next_state, done))

@tf.function
def train_step(states, actions, rewards, next_states, dones):
    # TD target: r + γ · max_a Q_target(s', a)   (0 if terminal)
    next_q   = target_net(next_states, training=False)
    max_next = tf.reduce_max(next_q, axis=1)
    targets  = rewards + GAMMA * max_next * (1.0 - dones)

    with tf.GradientTape() as tape:
        q_values = online_net(states, training=True)
        # Select only the Q-value for the action that was actually taken
        actions_oh = tf.one_hot(actions, NUM_ACTIONS)
        predicted  = tf.reduce_sum(q_values * actions_oh, axis=1)
        loss = tf.reduce_mean(tf.square(targets - predicted))   # MSE

    grads = tape.gradient(loss, online_net.trainable_variables)
    optimizer.apply_gradients(zip(grads, online_net.trainable_variables))
    return loss

def sample_and_train():
    batch = random.sample(replay_buffer, BATCH_SIZE)
    states, actions, rewards, next_states, dones = zip(*batch)

    states      = tf.constant(np.array(states),      dtype=tf.float32)
    actions     = tf.constant(np.array(actions),     dtype=tf.int32)
    rewards     = tf.constant(np.array(rewards),     dtype=tf.float32)
    next_states = tf.constant(np.array(next_states), dtype=tf.float32)
    dones       = tf.constant(np.array(dones),       dtype=tf.float32)

    return train_step(states, actions, rewards, next_states, dones)


# ── Custom Reward Shaping ──────────────────────────────────────────────────────
class RewardShaper:
    """
    Tracks per-episode state to compute the composite shaping reward:

        R = ΔProgress × SCREEN_BONUS
          + GameComplete × (COMPLETION_BONUS − ALPHA × t)
          − BETA × 1                          (per-step time penalty)
          − STALL_PENALTY  (if steps_since_progress ≥ STALL_THRESHOLD)

    All shaping terms (excluding the raw env reward) are clipped to
    [-REWARD_CLIP, +REWARD_CLIP] before being added, which keeps
    gradients stable without discarding the sign of the signal.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.current_screen      = 0      # Proxy for game progress (lives-based)
        self.steps_since_progress = 0     # Stall detector counter
        self.episode_steps       = 0      # Total steps this episode

    def shape(self, raw_reward: float, info: dict, terminated: bool, truncated: bool) -> float:
        """
        Compute the shaped reward for one transition.

        Parameters
        ----------
        raw_reward  : reward returned by the environment
        info        : info dict from env.step()
        terminated  : True if the episode ended naturally (e.g. game over)
        truncated   : True if the episode was cut short by a time limit
        done        : terminated OR truncated

        Returns
        -------
        shaped_reward : float  (raw_reward + clipped shaping terms)
        """
        self.episode_steps += 1
        shaping = 0.0

        # 1. Per-timestep time penalty — constant pressure to move fast
        shaping -= BETA

        # 2. Progress detection via RAM / info
        #    King Kong exposes 'lives' in info; losing a life means we
        #    treat it as a screen reset rather than forward progress.
        #    Screen advancement is inferred from score jumps > threshold.
        new_screen = info.get("screen", self.current_screen)   # fallback: unchanged

        # Fallback heuristic: large positive raw reward ≈ stage clear
        if raw_reward >= 100:
            new_screen = self.current_screen + 1

        if new_screen > self.current_screen:
            shaping += SCREEN_BONUS
            self.current_screen       = new_screen
            self.steps_since_progress = 0
        else:
            self.steps_since_progress += 1

        # 3. Mid-screen checkpoint bonus (score-based heuristic)
        if 0 < raw_reward < 100:
            shaping += CHECKPOINT_BONUS
            self.steps_since_progress = 0   # scoring counts as progress

        # 4. Stall penalty — discourage getting stuck
        if self.steps_since_progress >= STALL_THRESHOLD:
            shaping -= STALL_PENALTY
            # Don't keep stacking: reset counter so penalty fires once per window
            self.steps_since_progress = 0

        # 5. Completion bonus — reward finishing quickly
        #    King Kong has no explicit "win" signal in ALE; approximate by
        #    reaching a high screen count without a game-over termination.
        if terminated and not truncated and self.current_screen >= 4:
            completion = COMPLETION_BONUS - ALPHA * self.episode_steps
            shaping += max(completion, 0.0)   # Never punish for slow completion

        # Clip shaping terms to stabilise gradients, then add raw env reward
        shaping = float(np.clip(shaping, -REWARD_CLIP, REWARD_CLIP))
        return raw_reward + shaping


# ── Evaluation ─────────────────────────────────────────────────────────────────
def evaluate(eval_seconds=EVAL_SECONDS):
    """
    Run the greedy policy (epsilon=0) for a fixed wall-clock window.
    Returns total reward accumulated across however many episodes fit.
    Uses raw env rewards only — shaping is a training aid, not a metric.
    """
    eval_env = gym.make("ALE/KingKong-v5", render_mode="rgb_array")
    frame, _ = eval_env.reset()
    total_reward = 0.0
    deadline     = time.time() + eval_seconds

    while time.time() < deadline:
        processed = preprocess_frame(eval_env.render())
        tensor    = processed[np.newaxis, ..., np.newaxis]
        logits    = online_net(tensor, training=False)
        action    = int(tf.argmax(logits[0]).numpy())

        _, reward, terminated, truncated, _ = eval_env.step(action)
        total_reward += reward      # Raw score — unshapen

        if terminated or truncated:
            frame, _ = eval_env.reset()     # Keep playing until time is up

    eval_env.close()
    return total_reward


# ── Training loop ──────────────────────────────────────────────────────────────
print("Filling replay buffer …")
frame       = env.render()
epsilon     = EPSILON_START
total_steps = 0
episode_num = 0

eval_scores  = []   # (wall_time, score) checkpoints
train_start  = time.time()
last_eval_at = train_start

shaper = RewardShaper()   # One shaper instance; reset at every episode boundary

# ── Phase 1: pre-fill replay buffer with random experience ─────────────────────
prefill_obs, _ = env.reset()
while len(replay_buffer) < MIN_REPLAY_SIZE:
    f = env.render()
    a = env.action_space.sample()
    _, r, term, trunc, _ = env.step(a)
    s  = preprocess_frame(f)[..., np.newaxis]
    f2 = env.render()
    s2 = preprocess_frame(f2)[..., np.newaxis]
    store_transition(s, a, r, s2, False)
    if term or trunc:
        env.reset()

print(f"Replay buffer filled ({len(replay_buffer)} transitions). Training …")

# ── Phase 2: main train + periodically-evaluate loop ──────────────────────────
obs, _ = env.reset()
shaper.reset()

while time.time() - train_start < TRAIN_SECONDS:
    frame  = env.render()
    state  = preprocess_frame(frame)[..., np.newaxis]   # (84,84,1)

    action = select_action(frame, epsilon)
    _, raw_reward, terminated, truncated, info = env.step(action)

    # ── Apply composite reward shaping ────────────────────────────────────────
    shaped_reward = shaper.shape(raw_reward, info, terminated, truncated)

    next_frame = env.render()
    next_state = preprocess_frame(next_frame)[..., np.newaxis]

    # Store the SHAPED reward — the agent learns from the shaped signal
    store_transition(state, action, shaped_reward, next_state,
                     float(terminated and not truncated))

    loss = sample_and_train()
    total_steps += 1

    # Sync target network
    if total_steps % TARGET_UPDATE_EVERY == 0:
        target_net.set_weights(online_net.get_weights())
        print(f"  [step {total_steps}] target network updated")

    if terminated or truncated:
        episode_num += 1
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        obs, _ = env.reset()
        shaper.reset()   # Reset progress tracking for the new episode

    # ── Periodic evaluation checkpoint ────────────────────────────────────────
    now = time.time()
    if now - last_eval_at >= EVAL_EVERY:
        elapsed = now - train_start
        print(f"\n{'─'*50}")
        print(f"Evaluating at t={elapsed:.0f}s  (ε={epsilon:.3f}, "
              f"steps={total_steps}, episodes={episode_num}) …")
        score = evaluate(EVAL_SECONDS)
        eval_scores.append((elapsed, score))
        print(f"  Eval score ({EVAL_SECONDS}s window): {score:.1f}")
        print(f"{'─'*50}\n")
        last_eval_at = time.time()   # Reset clock after eval overhead

print("\nTraining complete.")
print("\nPerformance over time:")
print(f"  {'Time (s)':>10}  {'Score':>10}")
for t, s in eval_scores:
    print(f"  {t:>10.0f}  {s:>10.1f}")

env.close()
cv2.destroyAllWindows()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 84, 84, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 20, 20, 16)     │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 9, 9, 32)       │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2592)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       331,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 341,942 (1.30 MB)

 Trainable params: 341,942 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

Networks created
Filling replay buffer …
Replay buffer filled (1000 transitions). Training …
  [step 500] target network updated
  [step 1000] target network updated
  [step 1500] target network updated
  [step 2000] target network updated
  [step 2500] target network updated
  [step 3000] target network updated
  [step 3500] target network updated

──────────────────────────────────────────────────
Evaluating at t=60s  (ε=0.990, steps=3929, episodes=2) …
  Eval score (30s window): 150.0
──────────────────────────────────────────────────

  [step 4000] target network updated
  [step 4500] target network updated
  [step 5000] target network updated
  [step 5500] target network updated
  [step 6000] target network updated
  [step 6500] target network updated
  [step 7000] target network updated
  [step 7500] target network updated

──────────────────────────────────────────────────
Evaluating at t=150s  (ε=0.980, steps=7720, episodes=4) …
  Eval score (30s window): 25.0
─────────────────

In [ ]:
""" TEST PERFORMANCE """

gym.register_envs(ale_py)

def preprocess_obs(obs):
    gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return (small / 255.0).astype(np.float32)

eval_env = gym.make("ALE/KingKong-v5", render_mode="human")
obs, _ = eval_env.reset()

deadline = time.time() + 60

while time.time() < deadline:
    tensor = preprocess_obs(obs)[np.newaxis, ..., np.newaxis]
    action = int(tf.argmax(online_net(tensor, training=False)[0]).numpy())

    obs, reward, terminated, truncated, _ = eval_env.step(action)

    if terminated or truncated:
        obs, _ = eval_env.reset()

eval_env.close()

: 

In [4]:
""" SAVE MODEL """

online_net.save("kingkong_dqn.keras")
print("Model saved to kingkong_dqn.keras")

Model saved to kingkong_dqn.keras
